# **Autogen RoundRobin Group Agent**
- A team that runs a group chat with participants taking turns in a round-robin fashion to publish a message to all.

In [1]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.ollama import OllamaChatCompletionClient
import asyncio
from dotenv import load_dotenv
import os
load_dotenv()

True

## **Importing Model**

In [2]:
ollama_model_clint = OllamaChatCompletionClient(model="llama3.1")

## **Define 3 specific Agent**
- dsa solver
- code reviewer
- code editor

In [7]:
dsa_solver = AssistantAgent(
    name="Complex_DSA_solver",
    model_client=ollama_model_clint,
    description="A DSA solver",
    system_message="You give code in python to solve complex dsa problems. Give under 100 words"
)


code_reviewer = AssistantAgent(
    name="Code_Reviewer",
    model_client=ollama_model_clint,
    description="A code Reviewer",
    system_message="You review the code given by the complex_dsa_solver and make sure it is optimized. Given under 20 words. If you think that the code is fine, Please say 'TERMINATE'"
)

code_editor = AssistantAgent(
    name="Code_Editor",
    model_client=ollama_model_clint,
    description="A code Editor.",
    system_message="YOu make the code easy to understand and add comments wherever required. Give under 20 words."
)

In [10]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

my_termination = TextMentionTermination(text="TERMINATE") #| MaxMessageTermination(max_messages=2)

team = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer, code_editor],
    termination_condition=my_termination,
    max_turns= 10
)

In [11]:
from autogen_agentchat.messages import TextMessage
async def run_team():
    task = TextMessage(content="Write a python code like find a median two sorted array.", source="user")
    
    result = await team.run(task=task)
    
    for each_agent_msg in result.messages:
        print(f"{each_agent_msg.source} :-> {each_agent_msg.content}")
    
await run_team()

user :-> Write a python code like find a median two sorted array.
Complex_DSA_solver :-> Here is the Python code for finding the median of two sorted arrays using binary search:

```python
def findMedian(arr1, arr2):
    if len(arr1) > len(arr2):
        arr1, arr2 = arr2, arr1
    
    total_len = len(arr1) + len(arr2)
    half_len = total_len // 2
    
    left = 0
    right = len(arr1) - 1
    
    while True:
        i = (left + right) // 2
        j = half_len - i - 2
        
        max_left_x = float('-inf') if i < 0 else arr1[i]
        min_right_x = float('inf') if i >= len(arr1) else arr1[i+1]
        
        max_left_y = float('-inf') if j < 0 else arr2[j]
        min_right_y = float('inf') if j >= len(arr2) else arr2[j+1]
        
        if max_left_x <= min_right_y and max_left_y <= min_right_x:
            if total_len % 2 == 0:
                return (max(max_left_x, max_left_y) + min(min_right_x, min_right_y)) / 2.0
            else:
                return max(max_le

## **Built-In Termination Conditions:**

- MaxMessageTermination: Stops after a specified number of messages have been produced, including both agent and task messages.

- TextMentionTermination: Stops when specific text or string is mentioned in a message (e.g., “TERMINATE”).

- TokenUsageTermination: Stops when a certain number of prompt or completion tokens are used. This requires the agents to report token usage in their messages.

- TimeoutTermination: Stops after a specified duration in seconds.

- HandoffTermination: Stops when a handoff to a specific target is requested. Handoff messages can be used to build patterns such as Swarm. This is useful when you want to pause the run and allow application or user to provide input when an agent hands off to them.

- SourceMatchTermination: Stops after a specific agent responds.

- ExternalTermination: Enables programmatic control of termination from outside the run. This is useful for UI integration (e.g., “Stop” buttons in chat interfaces).

- StopMessageTermination: Stops when a StopMessage is produced by an agent.

- TextMessageTermination: Stops when a TextMessage is produced by an agent.

- FunctionCallTermination: Stops when a ToolCallExecutionEvent containing a FunctionExecutionResult with a matching name is produced by an agent.

- FunctionalTermination: Stop when a function expression is evaluated to True on the last delta sequence of messages. This is useful for quickly create custom termination conditions that are not covered by the built-in ones.

## **Run Team as Streaming Mode**

In [12]:
from autogen_agentchat.base import TaskResult

task = "Write a python code that can reverse the input string."

async for message in team.run_stream(task=task):
    if isinstance(message, TaskResult):
        print("Stop Reason: ", message.stop_reason)
    else:
        print(message.source, message)

user id='35c391ca-64a8-4a7c-8839-5e8d40cc78ff' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 37, 23, 722724, tzinfo=datetime.timezone.utc) content='Write a python code that can reverse the input string.' type='TextMessage'
Code_Editor id='31422b0f-5a0e-40e0-a1a1-f16b1f6c87a0' source='Code_Editor' models_usage=RequestUsage(prompt_tokens=2662, completion_tokens=232) metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 37, 43, 496384, tzinfo=datetime.timezone.utc) content='Here is a simple Python function to reverse an input string:\n\n```python\ndef reverse_string(s):\n    return s[::-1]\n\n# Example usage:\ninput_str = "Hello World"\nprint(reverse_string(input_str))  # Output: dlroW olleH\n```\n\nIn this code, `s[::-1]` is using Python\'s slice notation to create a reversed copy of the input string. The `::-1` part means start at the end of the string and end at position 0, move with the step -1 which means one step backwards.\n\nHowever, i